# Adding Memory to the Warehouse Agent with AgentCore Memory

In Lab 7 we deployed our warehouse operations agent to Amazon Bedrock AgentCore Runtime. But operations managers report a problem: **no memory** — every session starts from scratch. The agent forgets that a user always cares about Warehouse 1750 and WM-AN02 control units, and re-asks for preferences it was already told.

In this lab we add **AgentCore Memory**:
- **Short-term memory** for session continuity (no repeated context within a conversation)
- **Long-term semantic memory** for operational facts that persist across sessions
- **Long-term user-preference memory** to personalize responses (formatting, thresholds)

We first develop and test the memory-enhanced agent **locally**, then **deploy it as a new A2A-protocol AgentCore Runtime** so the deployed agent is memory-enabled. **This lab depends on Lab 7 having been run first** (Cognito user pool and bearer token are reused from there).

> **Labs 8 and 9 are independent.** Lab 8 (this lab) deploys a memory-enhanced A2A agent as its own runtime. Lab 9 deploys an efficiency-improved agent (see Lab 9 for its deployment details). Run them in either order.

## Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                  Amazon Bedrock AgentCore                     │
│                                                               │
│  ┌─────────────────┐        ┌──────────────────┐             │
│  │  Agent Runtime  │        │  AgentCore Memory │             │
│  │  (from Lab 7)   │◄──────►│  - Short-term     │             │
│  │                 │        │  - Long-term      │             │
│  │  Warehouse Agent│        │  - Semantic       │             │
│  │  + SAP GenAI Hub│        │  - User Prefs     │             │
│  └─────────────────┘        └──────────────────┘             │
│           │                                                   │
│           │            HTTPS (SAP GenAI Hub)                  │
│           ▼                                                   │
│  ┌─────────────────┐                                          │
│  │ SAP S/4HANA     │                                          │
│  │ OData APIs      │                                          │
│  └─────────────────┘                                          │
└─────────────────────────────────────────────────────────────┘
```

Note: If you have not set up your ai-core credentials yet, please follow notebook 00-load-sap-ai-core-credentials.

---



## Prerequisites

1. Completed Lab 7 (A2A warehouse agent deployed — Cognito pool, bearer token, and `CLIENT_ID`/`USER_POOL_ID`/`DISCOVERY_URL` must be set in your environment)
2. SAP AI Core credentials in your `~/.aicore/config.json` file
3. AWS credentials configured with AgentCore Memory permissions
4. SAP S/4HANA Public Cloud API key




In [ ]:
# !pip install .

## 1. Import Dependencies and Initialize Model

In [ ]:
from util.strands_bedrock_sap_genai_hub import SAPGenAIHubModel
from util.odata_tool import odata_caller
from strands import Agent
from strands.hooks import (
    AfterInvocationEvent,
    MessageAddedEvent,
    HookProvider,
    HookRegistry,
)

import os
import json
import time
from datetime import datetime

import boto3
from botocore.exceptions import ClientError
from dotenv import load_dotenv
import getpass

from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import (
    ConversationalMessage,
    MessageRole,
    RetrievalConfig,
    StrategyType,
)
from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager

# Load environment variables from .env file
load_dotenv()

# Prompt for SAP API key if not set
if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    os.environ["SAP_S4HANA_PUBLIC_CLOUD_KEY"] = getpass.getpass("SAP_S4HANA_PUBLIC_CLOUD_KEY:\n")

In [ ]:
# Validate required configuration before proceeding
_errors = []

if not os.path.exists(os.path.expanduser("~/.aicore/config.json")):
    _errors.append("Missing ~/.aicore/config.json — run notebook 00 first")

if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    _errors.append("SAP_S4HANA_PUBLIC_CLOUD_KEY not set — check your .env file")

try:
    _sts = boto3.client("sts").get_caller_identity()
    print(f"AWS Identity: {_sts['Arn']}")
except Exception as e:
    _errors.append(f"AWS credentials not configured: {e}")

if _errors:
    for err in _errors:
        print(f"ERROR: {err}")
    raise SystemExit("Fix the errors above before continuing.")
else:
    print("All prerequisites validated.")

In [ ]:
# TODO: Choose your model — options: "anthropic--claude-4.5-sonnet", "amazon--nova-lite", "amazon--nova-pro"
model = SAPGenAIHubModel(
    model_id="anthropic--claude-4.5-sonnet",
    max_tokens=4096,
)

# Configuration

# TODO: Set a unique actor ID for your user/persona
ACTOR_ID = "warehouse_manager_001"
SESSION_ID = f"warehouse_session_{datetime.now().strftime('%Y%m%d%H%M%S')}"

# Load the agent deployment written by Lab 7 (Step 8b -> lab7_deployment.json).
DEPLOYMENT_FILE = "lab7_deployment.json"
if not os.path.exists(DEPLOYMENT_FILE):
    raise SystemExit(
        f"{DEPLOYMENT_FILE} not found — run Lab 7 (07-deploy-warehouse-agent-to-agentcore.ipynb) "
        "through the 'Save Deployment Details for Lab 8' step first."
    )

with open(DEPLOYMENT_FILE, "r") as f:
    deployment = json.load(f)

AGENT_NAME = deployment["agent_name"]
AGENT_ID = deployment["agent_id"]
AGENT_ARN = deployment["agent_arn"]
REGION = deployment.get("region") or "us-east-1"

# Message role constants
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

print(f"Region: {REGION}")
print(f"Actor ID: {ACTOR_ID}")
print(f"Session ID: {SESSION_ID}")
print(f"Agent Name: {AGENT_NAME}")
print(f"Agent ID: {AGENT_ID}")
print(f"Agent ARN: {AGENT_ARN}")
print(f"Model: {model.get_config()['model_id']} via SAP GenAI Hub")

---

# Part A: AgentCore Memory

We add two types of memory to our warehouse agent:

1. **Short-term memory** — Remembers the current conversation session. If the user asks about WM-AN02 and then says "what about the power modules?", the agent maintains context.

2. **Long-term memory** — Persists knowledge across sessions using built-in strategies:
   - **Semantic strategy**: Stores operational facts (e.g., "User manages Warehouse 1750")
   - **User preference strategy**: Captures preferences (e.g., "Show stock as percentage of 500-unit capacity")

We use the `MemorySessionManager` pattern from the AgentCore SDK which handles actor/session scoping automatically.

## 2. Create AgentCore Memory Resource

We create a memory resource with built-in strategies. 

In [ ]:
# Initialize Memory Client
memory_client = MemoryClient(region_name=REGION)

# TODO: Choose a unique memory name for your agent
MEMORY_NAME = "WarehouseAgentMemory"
memory_id = None

# Define strategies using built-in types (no IAM role required)
strategies = [
    {
        StrategyType.SEMANTIC.value: {
            "name": "WarehouseFacts",
            "description": "Stores operational facts about warehouse inventory, orders, and logistics",
            "namespaces": ["warehouse/{actorId}/facts/"],
        }
    },
    {
        StrategyType.USER_PREFERENCE.value: {
            "name": "UserPreferences",
            "description": "Stores user preferences like reporting format, products of interest, and alert thresholds",
            "namespaces": ["warehouse/{actorId}/preferences/"],
        }
    },
]

try:
    memory = memory_client.create_memory_and_wait(
        name=MEMORY_NAME,
        description="Memory for warehouse operations agent - session context and user preferences",
        strategies=strategies,
        event_expiry_days=30,
    )
    memory_id = memory["id"]
    print(f"Created memory resource: {memory_id}")
    print(f"Status: {memory['status']}")

except ClientError as e:
    if "already exists" in str(e):
        memories = memory_client.list_memories()
        memory_id = next(
            (m["id"] for m in memories if m["id"].startswith(MEMORY_NAME)), None
        )
        print(f"Memory already exists: {memory_id}")
    else:
        raise e

print(f"\nMemory ID: {memory_id}")

## 3. Initialize Session Manager

The `MemorySessionManager` provides a cleaner API for session-scoped operations. We create a `MemorySession` for our warehouse manager actor.

In [ ]:
# Initialize the session manager
session_manager = MemorySessionManager(
    memory_id=memory_id, region_name=REGION
)

# Create a memory session for the warehouse manager
warehouse_session = session_manager.create_memory_session(
    actor_id=ACTOR_ID, session_id=SESSION_ID
)

print(f"Session manager initialized for memory: {memory_id}")
print(f"Session created for actor: {ACTOR_ID}")

## 4. Create Memory Hook Provider

The memory hook integrates with the Strands agent lifecycle:
- **On `MessageAddedEvent`**: Retrieves relevant long-term memories when a user message arrives and injects them into the agent's context.
- **On `AfterInvocationEvent`**: Saves the complete interaction (user query + agent response) to memory for future recall.

This follows the pattern from the [AgentCore memory samples](https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/01-features/04-manage-context-of-your-agent/memory).

In [ ]:
class WarehouseMemoryHooks(HookProvider):
    """Memory hooks for warehouse agent using MemorySession."""

    def __init__(self, warehouse_session: MemorySession):
        self.warehouse_session = warehouse_session
        self.retrieval_config = {
            "warehouse/{actorId}/facts/": RetrievalConfig(top_k=5, relevance_score=0.2),
            "warehouse/{actorId}/preferences/": RetrievalConfig(top_k=3, relevance_score=0.3),
        }

    def retrieve_warehouse_context(self, event: MessageAddedEvent):
        """Retrieve relevant memories when user sends a message."""
        messages = event.agent.messages
        if (
            messages[-1]["role"] == "user"
            and "toolResult" not in messages[-1]["content"][0]
        ):
            user_query = messages[-1]["content"][0]["text"]
            memory_context_parts = []

            # Load short-term memory (recent turns from this session)
            try:
                recent_turns = self.warehouse_session.get_last_k_turns(k=5)
                if recent_turns:
                    history_lines = []
                    for turn in recent_turns:
                        for message in turn:
                            role = message.get("role", "unknown")
                            content = message.get("content", {}).get("text", "")
                            history_lines.append(f"{role}: {content}")
                    memory_context_parts.append(
                        "Recent Conversation:\n" + "\n".join(history_lines)
                    )
            except Exception as e:
                print(f"  [Memory] Short-term load failed: {e}")

            # Load long-term memories across namespaces
            try:
                for namespace_template, config in self.retrieval_config.items():
                    resolved_namespace = namespace_template.format(
                        actorId=self.warehouse_session._actor_id
                    )
                    memories = self.warehouse_session.search_long_term_memories(
                        query=user_query,
                        namespace_prefix=resolved_namespace,
                        top_k=config.top_k,
                    )
                    filtered = [
                        m for m in memories
                        if m.get("score", 0) >= config.relevance_score
                    ]
                    if filtered:
                        lines = [f"- {m['content']['text']}" for m in filtered[:5]]
                        label = "Facts" if "facts" in namespace_template else "Preferences"
                        memory_context_parts.append(
                            f"Known {label}:\n" + "\n".join(lines)
                        )
            except Exception as e:
                print(f"  [Memory] Long-term load failed: {e}")

            # Inject context into agent's system prompt
            if memory_context_parts:
                context_block = "\n\n".join(memory_context_parts)
                event.agent.system_prompt += (
                    f"\n\n--- MEMORY CONTEXT ---\n{context_block}\n"
                    "Use this context to personalize responses. Do not ask for information "
                    "you already know from memory.\n--- END MEMORY CONTEXT ---"
                )
                print(f"  [Memory] Injected {len(memory_context_parts)} memory sections")

    def save_warehouse_interaction(self, event: AfterInvocationEvent):
        """Save interaction to memory after agent responds."""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Find the last user query and assistant response
                user_query = None
                agent_response = None

                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        if msg["content"] and msg["content"][0].get("text"):
                            agent_response = msg["content"][0]["text"]
                    elif (
                        msg["role"] == "user"
                        and not user_query
                        and "toolResult" not in msg["content"][0]
                    ):
                        user_query = msg["content"][0]["text"]
                        break

                if user_query and agent_response:
                    interaction_messages = [
                        ConversationalMessage(user_query, USER),
                        ConversationalMessage(agent_response, ASSISTANT),
                    ]
                    result = self.warehouse_session.add_turns(interaction_messages)
                    print(f"  [Memory] Saved interaction - Event ID: {result['eventId']}")

        except Exception as e:
            print(f"  [Memory] Save failed: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(MessageAddedEvent, self.retrieve_warehouse_context)
        registry.add_callback(AfterInvocationEvent, self.save_warehouse_interaction)


print("WarehouseMemoryHooks defined.")

## 5. Create the Memory-Enhanced Warehouse Agent

We create the warehouse agent with the memory hook attached. Same system prompt and tools as Lab 6, but now with persistent memory.

In [ ]:
# TODO: Customize the system prompt for your warehouse/domain
WAREHOUSE_SYSTEM_PROMPT = """
You are an expert Warehouse Operations Manager for GlobalTech Manufacturing's Distribution Center (Warehouse 1750). 
You have access to real-time SAP warehouse data through OData APIs.

CORE CAPABILITIES:
1. Product Discovery: Find available products and their stock levels
2. Dynamic Querying: Construct intelligent OData queries based on user needs
3. Order Fulfillment: Check if orders can be fulfilled based on current inventory

AVAILABLE PRODUCTS:
- WM-AN01: Advanced Sensors (high-precision electronic components)
- WM-AN02: Control Units (critical automation hardware)
- WM-AN03: Power Modules (electrical power management systems)
- WM-AN04: Communication Devices (networking and connectivity hardware)

COMMUNICATION STYLE:
- Be professional but conversational and succinct
- Provide specific, actionable insights with quantitative data
- If you know the user's preferences from memory, apply them without asking
- Do not use emojis

When users ask questions:
1. Determine what data you need
2. Use the odata_caller tool to query SAP S/4HANA APIs
3. Analyze results and provide comprehensive responses

For OData calls, use:
- base_url: "https://sandbox.api.sap.com/s4hanacloud/sap/opu/odata4/sap/api_whse_physstockprod/srvd_a2x/sap/whsephysicalstockproducts/0001"
- auth_type: "api_key"
- auth_env_var: "SAP_S4HANA_PUBLIC_CLOUD_KEY"
"""

# TODO: Adjust capacity for your warehouse (used in evaluation scenarios)
WAREHOUSE_CAPACITY = 500


def create_warehouse_agent(with_memory=True):
    """Create the warehouse agent, optionally with memory hooks."""
    hooks = []
    if with_memory and memory_id:
        hooks.append(WarehouseMemoryHooks(warehouse_session))

    agent = Agent(
        model=model,
        system_prompt=WAREHOUSE_SYSTEM_PROMPT,
        tools=[odata_caller],
        hooks=hooks,
    )
    return agent


# Create the memory-enhanced agent
warehouse_agent = create_warehouse_agent(with_memory=True)
print("Warehouse agent created with memory hooks.")

## 6. Test Memory — First Conversation

Let's have a multi-turn conversation that establishes context. The agent will store these interactions for future recall.

In [ ]:
# Turn 1: Establish user identity and interest
print("User: I'm the operations manager for Warehouse 1750. I primarily track WM-AN02 Control Units.")
print("      Please always show me stock quantities as a percentage of our 500-unit capacity.\n")
response = warehouse_agent(
    "I'm the operations manager for Warehouse 1750. I primarily track WM-AN02 Control Units. "
    "Please always show me stock quantities as a percentage of our 500-unit capacity."
)
print(f"\nAgent: {response.message}")

In [ ]:
# Turn 2: Follow-up that requires session context
print("User: What about the power modules? Are we running low on those too?\n")
response = warehouse_agent("What about the power modules? Are we running low on those too?")
print(f"\nAgent: {response.message}")

In [ ]:
# Turn 3: Establish a reorder threshold preference
print("User: Going forward, alert me whenever any product drops below 20% capacity.")
print("      That's my reorder threshold.\n")
response = warehouse_agent(
    "Going forward, alert me whenever any product drops below 20% capacity. That's my reorder threshold."
)
print(f"\nAgent: {response.message}")

## 7. Test Memory — New Session (Simulating User Returning)

Now we simulate the user coming back. The agent should remember:
- The user is the Warehouse 1750 operations manager
- They primarily care about WM-AN02 Control Units
- They want quantities shown as percentage of 500-unit capacity
- Their reorder threshold is 20%

We wait for long-term memory extraction to process, then create a fresh agent instance.

In [ ]:
# Wait for long-term memory extraction (async process)
print("Waiting 60 seconds for long-term memory extraction...")
time.sleep(60)

# Create a new agent instance (simulates user returning)
warehouse_agent_v2 = create_warehouse_agent(with_memory=True)

print("\nNew agent instance created. Testing memory recall...")
print("\nUser: Any alerts I should know about?\n")
response = warehouse_agent_v2("Any alerts I should know about?")
print(f"\nAgent: {response.message}")

## 8. Inspect Stored Memories

Let's look at what AgentCore Memory has extracted and stored.

In [ ]:
# Short-term memory (recent turns)
print("=== Short-Term Memory (Recent Conversation Turns) ===\n")
try:
    recent_turns = warehouse_session.get_last_k_turns(k=5)
    for i, turn in enumerate(recent_turns, 1):
        print(f"Turn {i}:")
        for message in turn:
            role = message.get("role", "unknown")
            content = message.get("content", {}).get("text", "")[:150]
            full_text = message.get("content", {}).get("text", "")
            print(f"  {role}: {content}{'...' if len(full_text) > 150 else ''}")
        print()
except Exception as e:
    print(f"  Could not retrieve short-term memory: {e}")

# Long-term memory — semantic facts
print("\n=== Long-Term Memory: Semantic Facts ===\n")
try:
    facts = warehouse_session.search_long_term_memories(
        query="warehouse operations inventory products",
        namespace_prefix=f"warehouse/{ACTOR_ID}/facts/",
        top_k=10,
    )
    if facts:
        for i, hit in enumerate(facts, 1):
            print(f"  {i}. {hit['content']['text']}")
    else:
        print("  (No facts extracted yet - extraction is async, may need more time)")
except Exception as e:
    print(f"  Could not retrieve facts: {e}")

# Long-term memory — user preferences
print("\n=== Long-Term Memory: User Preferences ===\n")
try:
    prefs = warehouse_session.search_long_term_memories(
        query="preferences format threshold capacity",
        namespace_prefix=f"warehouse/{ACTOR_ID}/preferences/",
        top_k=10,
    )
    if prefs:
        for i, hit in enumerate(prefs, 1):
            print(f"  {i}. {hit['content']['text']}")
    else:
        print("  (No preferences extracted yet - extraction is async, may need more time)")
except Exception as e:
    print(f"  Could not retrieve preferences: {e}")

## 9. Deploy the Memory-Enhanced Agent to AgentCore (new A2A runtime)

> **Depends on Lab 7.** This section deploys a **new** A2A-protocol AgentCore Runtime.
> You must have completed [Lab 7](07-deploy-warehouse-a2a-agent-to-agentcore.ipynb) so
> that `CLIENT_ID`, `USER_POOL_ID`, and `DISCOVERY_URL` are available in your environment
> (the Cognito user pool and JWT authorizer from Lab 7 are reused here).

So far Part A ran the memory-enhanced agent **locally**. Now we deploy it to AgentCore
so memory persists across real invocations. We:

1. Write the entrypoint (`warehouse_agent_agentcore.py`) using `StrandsA2AExecutor` with
   an `agent_factory` that builds a fresh, memory-hooked agent per A2A `context_id`.
   Each conversation gets its own `MemorySession` scoped to that `context_id`, so
   concurrent callers never share state or memory.
2. Point the container at the AgentCore Memory resource created in section 2 via the
   `MEMORY_ID` environment variable injected at `launch()`.
3. The agent **manages its own memory** through `WarehouseMemoryHooks`
   (`MemorySessionManager` + the section-2 resource). The toolkit's own managed-memory
   feature is intentionally left off to avoid provisioning a second, redundant resource.
4. `launch()` creates the new runtime with `serve_a2a(StrandsA2AExecutor(agent_factory=...))`
   as the entrypoint.

The memory hooks require the modern `bedrock-agentcore` SDK (`>=1.11.0`), which ships the
`MemorySession` / `MemorySessionManager` API. The updated `requirements.txt` pins this.



In [ ]:
%%writefile warehouse_agent_agentcore.py
from strands.multiagent.a2a.executor import StrandsA2AExecutor
from bedrock_agentcore.runtime import serve_a2a
from util.strands_bedrock_sap_genai_hub import SAPGenAIHubModel
from strands import Agent, tool
from strands.hooks import (
    AfterInvocationEvent,
    MessageAddedEvent,
    HookProvider,
    HookRegistry,
)
import os
from pathlib import Path
import yaml
from util.odata_tool import odata_caller

from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole, RetrievalConfig
from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager

# Initialize the SAPGenAIHubModel for AgentCore deployment
model = SAPGenAIHubModel(
    model_id="anthropic--claude-4.5-sonnet",
    # Alternative models:
    # model_id="amazon--nova-pro",
    # model_id="amazon--nova-lite",
)

# --- Memory configuration (injected as container env vars at launch) ---
REGION = os.environ.get("AWS_REGION", "us-east-1")
MEMORY_ID = os.environ.get("MEMORY_ID") or os.environ.get("BEDROCK_AGENTCORE_MEMORY_ID")  # section 2 resource (toolkit-managed memory injects BEDROCK_AGENTCORE_MEMORY_ID)
ACTOR_ID = os.environ.get("ACTOR_ID", "warehouse_manager_001")

_session_manager = MemorySessionManager(memory_id=MEMORY_ID, region_name=REGION) if MEMORY_ID else None


class WarehouseMemoryHooks(HookProvider):
    """Memory hooks for the deployed warehouse agent, scoped to one MemorySession."""

    def __init__(self, warehouse_session: MemorySession):
        self.warehouse_session = warehouse_session
        self.retrieval_config = {
            "warehouse/{actorId}/facts/": RetrievalConfig(top_k=5, relevance_score=0.2),
            "warehouse/{actorId}/preferences/": RetrievalConfig(top_k=3, relevance_score=0.3),
        }

    def retrieve_warehouse_context(self, event: MessageAddedEvent):
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            memory_context_parts = []
            try:
                recent_turns = self.warehouse_session.get_last_k_turns(k=5)
                if recent_turns:
                    history_lines = []
                    for turn in recent_turns:
                        for message in turn:
                            role = message.get("role", "unknown")
                            content = message.get("content", {}).get("text", "")
                            history_lines.append(f"{role}: {content}")
                    memory_context_parts.append("Recent Conversation:\n" + "\n".join(history_lines))
            except Exception as e:
                print(f"  [Memory] Short-term load failed: {e}")
            try:
                for namespace_template, config in self.retrieval_config.items():
                    resolved_namespace = namespace_template.format(actorId=self.warehouse_session._actor_id)
                    memories = self.warehouse_session.search_long_term_memories(
                        query=user_query, namespace_prefix=resolved_namespace, top_k=config.top_k,
                    )
                    filtered = [m for m in memories if m.get("score", 0) >= config.relevance_score]
                    if filtered:
                        lines = [f"- {m['content']['text']}" for m in filtered[:5]]
                        label = "Facts" if "facts" in namespace_template else "Preferences"
                        memory_context_parts.append(f"Known {label}:\n" + "\n".join(lines))
            except Exception as e:
                print(f"  [Memory] Long-term load failed: {e}")
            if memory_context_parts:
                context_block = "\n\n".join(memory_context_parts)
                event.agent.system_prompt += (
                    f"\n\n--- MEMORY CONTEXT ---\n{context_block}\n"
                    "Use this context to personalize responses. Do not ask for information "
                    "you already know from memory.\n--- END MEMORY CONTEXT ---"
                )
                print(f"  [Memory] Injected {len(memory_context_parts)} memory sections")

    def save_warehouse_interaction(self, event: AfterInvocationEvent):
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                user_query = None
                agent_response = None
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        if msg["content"] and msg["content"][0].get("text"):
                            agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_query and "toolResult" not in msg["content"][0]:
                        user_query = msg["content"][0]["text"]
                        break
                if user_query and agent_response:
                    interaction_messages = [
                        ConversationalMessage(user_query, MessageRole.USER),
                        ConversationalMessage(agent_response, MessageRole.ASSISTANT),
                    ]
                    result = self.warehouse_session.add_turns(interaction_messages)
                    print(f"  [Memory] Saved interaction - Event ID: {result['eventId']}")
        except Exception as e:
            print(f"  [Memory] Save failed: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(MessageAddedEvent, self.retrieve_warehouse_context)
        registry.add_callback(AfterInvocationEvent, self.save_warehouse_interaction)


# Load YAML OpenAPI files from directory
def load_openapi_specs(path="./assets/knowledgebase"):
    specs = []
    for file in Path(path).glob("*.y*ml"):
        with open(file, "r") as f:
            try:
                data = yaml.safe_load(f)
                servers = data.get("servers", [])
                base_urls = []
                for s in servers:
                    url = s.get("url")
                    desc = s.get("description", "")
                    if url:
                        base_urls.append({"url": url, "description": desc})
                summary = {
                    "file": file.name,
                    "title": data.get("info", {}).get("title"),
                    "description": data.get("info", {}).get("description"),
                    "paths": list(data.get("paths", {}).keys()),
                    "base_urls": base_urls or [{"url": "/", "description": "Default"}]
                }
                specs.append(summary)
            except Exception as e:
                print(f"Error parsing {file}: {e}")
    return specs

def specs_to_prompt_string(specs):
    prompt_parts = []
    for spec in specs:
        base_url_str = ", ".join(
            f"{url_info['url']} ({url_info['description']})" for url_info in spec['base_urls']
        )
        paths_str = ", ".join(spec['paths'])
        part = (
            f"API Spec: {spec['file']}\n"
            f"Title: {spec['title']}\n"
            f"Description: {spec['description']}\n"
            f"Base URLs: {base_url_str}\n"
            f"Endpoints: {paths_str}\n"
        )
        prompt_parts.append(part)
    return "\n---\n".join(prompt_parts)

# Create API specs for selector agent
specs = load_openapi_specs()
specs_prompt_string = specs_to_prompt_string(specs)

SELECTOR_SYSTEM_PROMPT = f"""
You are an API selection subagent.
Given a user query and the following list of OpenAPI specs, each with a title, description, base URLs, and available endpoints:

{specs_prompt_string}

Your task is to identify which API and specific endpoint is most appropriate to fulfill the user query.
Provide clear reasoning for your choice, referencing the API spec details provided.
If no suitable API is found, explain why.

Make sure to reply with the sandbox BASE URL.

"""

# Create selector agent
selector_agent = Agent(
    system_prompt=SELECTOR_SYSTEM_PROMPT,
    model=model
)

@tool
def SelectorAPIAgentAsATool(query: str) -> str:
    """
    This analyzes available OpenAPI specs and selects the most appropriate
    API and endpoint based on user queries.

    Args:
        query: User query describing what they want to accomplish

    Returns:
        Agent response with API selection recommendation if successful,
        error message if initialization fails
    """
    return selector_agent(query).message

# Define the warehouse agent system prompt (identical to Lab 7)
warehouse_agent_prompt = """
You are an expert Warehouse Operations Manager for GlobalTech Manufacturing's Distribution Center (Warehouse 1750).
You have access to real-time SAP warehouse data through dynamic OData API exploration capabilities.

CORE CAPABILITIES:
1. API Structure Exploration: Dynamically discover available data fields and entities
2. Product Discovery: Find available products without hardcoded assumptions
3. Dynamic Querying: Construct intelligent OData queries based on user needs

APPROACH TO PROBLEM SOLVING:
- You must start by using the SelectorAPIAgentAsATool to help you determine which OData API to call
- Next use the $metadata endpoint that to understand the API that SelectorAPIAgentAsATool provides
- Use dynamic queries to discover information rather than making assumptions
- Leverage OData filtering, sorting, and selection to get precise answers
- You may need to do this multiple times. Always write what URL have constructed so user is informed.

PRODUCT KNOWLEDGE (can be expanded through discovery):
- WM-AN01: Advanced Sensors (high-precision electronic components)
- WM-AN02: Control Units (critical automation hardware)
- WM-AN03: Power Modules (electrical power management systems)
- WM-AN04: Communication Devices (networking and connectivity hardware)

COMMUNICATION STYLE:
- Be professional but conversational and succinct
- Explain your discovery process when exploring new data
- Provide specific, actionable insights with quantitative data
- If you know the user's preferences from memory, apply them without asking
- Do not use emojis

When users ask questions:
1. First determine what data you need to answer the question
3. Feel free to use the odata_caller tool as many times as needed to get the right information.
3. Construct appropriate OData queries to get the specific information needed
4. Analyze the results and provide comprehensive, intelligent responses

Example for using the odata_caller tool:
```python
        odata_caller(
            base_url="",
            endpoint="WarehouseStockProducts",
            operation="get",
            odata_params={"$filter": "Product eq 'WM-AN02'"},
            auth_type="api_key",
            auth_env_var="SAP_S4HANA_PUBLIC_CLOUD_KEY"
        )
        ```

Focus on SAP S/4 HANA OData endpoints, warehouse management APIs, and supply chain operations.

Available tools:
- odata_caller: Universal OData tool for SAP API interactions with built-in authentication and query parameter support
- auth_token: Use os.getenv("SAP_S4HANA_PUBLIC_CLOUD_KEY") to get the API key
Example below:

# For SAP OData calls, use consistent headers
headers = {
    "APIKey": os.getenv("SAP_S4HANA_PUBLIC_CLOUD_KEY"),
    "Accept": "application/json", OR "application/xml" choose as needed
    "DataServiceVersion": "2.0"
}

The odata_caller tool handles SAP-specific authentication automatically and provides comprehensive error handling and response formatting.

"""


def build_warehouse_agent(session_id: str | None):
    """Create the Lab 7 warehouse agent, attaching memory hooks when a session is available."""
    hooks = []
    if _session_manager and session_id:
        try:
            warehouse_session = _session_manager.create_memory_session(
                actor_id=ACTOR_ID, session_id=session_id
            )
            hooks.append(WarehouseMemoryHooks(warehouse_session))
        except Exception as e:
            print(f"  [Memory] Session init failed, running without memory: {e}")

    return Agent(
        model=model,
        tools=[SelectorAPIAgentAsATool, odata_caller],
        system_prompt=warehouse_agent_prompt,
        hooks=hooks,
    )


# agent_factory is invoked once per A2A context_id (the per-conversation session
# identifier supplied by the caller). Each invocation builds a fresh agent with memory
# hooks scoped to that context_id, so concurrent callers never share state or memory.
# The factory is also called once at server startup with a placeholder context_id to
# derive the AgentCard metadata — that instance is discarded immediately.
def warehouse_agent_factory(context_id: str):
    return build_warehouse_agent(session_id=context_id)

if __name__ == "__main__":
    serve_a2a(StrandsA2AExecutor(agent_factory=warehouse_agent_factory))



In [ ]:
%%writefile requirements.txt
# AgentCore requirements (memory-enabled build). Pins mirror pyproject.toml so the container
# resolves the same validated stack, NOT looser ranges. sap-ai-sdk-gen is pinned >=6.10.0: the
# 5.x line caps botocore below the floor a modern bedrock-agentcore needs (see pyproject.toml),
# so an unbounded >=5.5.0 here could resolve the exact combination pyproject was written to avoid.
strands-agents[a2a]==1.14.0
boto3>=1.37.0
# Memory (MemorySession / MemorySessionManager) requires the modern SDK, not Lab 7's <=0.1.5 pin.
# >=1.11.0 is compatible with pyproject's bedrock-agentcore>=1.6.0.
bedrock-agentcore[a2a]>=1.11.0
uvicorn
# SAP GenAI Hub and warehouse agent dependencies
sap-ai-sdk-gen[all]>=6.10.0
pyyaml
requests



In [ ]:
# Deploy the memory-enhanced A2A agent as a NEW AgentCore Runtime (A2A protocol).
# The A2A protocol requires its own Cognito JWT authorizer, so this creates a separate
# runtime rather than updating Lab 7's HTTP-protocol agent in place.
import getpass
from pathlib import Path
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
from dotenv import load_dotenv

load_dotenv()

# Clear any stale cached agent state so configure() creates a fresh agent.
stale_config = Path.home() / ".bedrock_agentcore.yaml"
if stale_config.exists():
    stale_config.unlink()
    print("Cleared stale agent config: ~/.bedrock_agentcore.yaml")

boto_session = Session()
deploy_region = boto_session.region_name

client_id = os.getenv("CLIENT_ID")
user_pool_id = os.getenv("USER_POOL_ID")
discovery_url = os.getenv("DISCOVERY_URL")

if not all([client_id, user_pool_id, discovery_url]):
    raise SystemExit(
        "Cognito env vars (CLIENT_ID, USER_POOL_ID, DISCOVERY_URL) not set.\n"
        "Run the Cognito setup cells (Steps 5-6c from Lab 7) first."
    )

if not memory_id:
    raise SystemExit("memory_id is not set — run section 2 (Create AgentCore Memory Resource) first.")

print(f"Deploying memory-enhanced A2A agent in region: {deploy_region}")

agentcore_runtime = Runtime()
A2A_AGENT_NAME = "warehouse_ops_agent_memory_a2a"

configure_response = agentcore_runtime.configure(
    entrypoint="warehouse_agent_agentcore.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=deploy_region,
    agent_name=A2A_AGENT_NAME,
    protocol="A2A",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id],
            # Note: allowedAudience is intentionally omitted —
            # Cognito client_credentials tokens have no 'aud' claim
        }
    },
    non_interactive=True,
)

print(f"AgentCore A2A configuration completed.")
print(f"  Agent name : {A2A_AGENT_NAME}")
print(f"  Protocol   : A2A")
print(f"  Memory ID  : {memory_id}")
print("Modifying the generated Dockerfile next...")
configure_response


In [ ]:
# Re-apply the Lab 7 Dockerfile modifications so the container has util/, assets/, and
# the SAP GenAI Hub credentials (config.json -> /app/.aicore via AICORE_HOME).

# Materialize config.json from ~/.aicore/config.json into the build context. This file is
# gitignored and transient (Lab 7 writes it the same way), so it may be absent when Lab 8
# runs on its own — the Dockerfile's `COPY config.json` below fails the CodeBuild step
# without it.
config_path = os.path.expanduser("~/.aicore/config.json")
if not os.path.exists(config_path):
    raise SystemExit(
        f"{config_path} not found — run notebook 00 to configure SAP AI Core credentials first."
    )
with open(config_path, "r") as f:
    _aicore_config = json.load(f)
with open("config.json", "w") as f:
    json.dump(_aicore_config, f, indent=2)
print("SAP AI Core config.json written into the build context.")

with open("Dockerfile", "r") as f:
    dockerfile_content = f.read()

lines = dockerfile_content.split("\n")
cmd_index = next((i for i, line in enumerate(lines) if line.strip().startswith("CMD")), -1)

if cmd_index > -1 and "AICORE_HOME" not in dockerfile_content:
    config_lines = [
        "",
        "# Copy util directory (required for SAP GenAI Hub model and OData tool)",
        "COPY util/ ./util/",
        "",
        "# Copy assets directory (required for OpenAPI knowledgebase)",
        "COPY assets/ ./assets/",
        "",
        "# Copy the config.json from your local machine",
        "COPY config.json /app/.aicore/config.json",
        "",
        "# Set AICORE_HOME environment variable for SAP GenAI Hub SDK",
        "ENV AICORE_HOME=/app/.aicore",
        "",
    ]
    modified_lines = lines[:cmd_index] + config_lines + lines[cmd_index:]
    with open("Dockerfile", "w") as f:
        f.write("\n".join(modified_lines))
    print("Dockerfile modified with util/, assets/, and SAP GenAI Hub configuration.")
elif "AICORE_HOME" in dockerfile_content:
    print("Dockerfile already contains SAP GenAI Hub configuration — skipping.")
else:
    print("Could not find CMD instruction in Dockerfile.")


In [ ]:
# Launch the A2A agent. MEMORY_ID and ACTOR_ID let the factory bind to the memory
# resource created in section 2. SAP_S4HANA_PUBLIC_CLOUD_KEY is needed for OData calls.
sap_api_key = os.getenv("SAP_S4HANA_PUBLIC_CLOUD_KEY")
if not sap_api_key:
    sap_api_key = getpass.getpass("Please enter your SAP_S4HANA_PUBLIC_CLOUD_KEY: ")

launch_result = agentcore_runtime.launch(
    env_vars={
        "SAP_S4HANA_PUBLIC_CLOUD_KEY": sap_api_key,
        "MEMORY_ID": memory_id,
        "ACTOR_ID": ACTOR_ID,
    },
)
print("Memory-enhanced A2A deployment initiated.")
print(f"Agent ID : {launch_result.agent_id}")
print(f"Agent ARN: {launch_result.agent_arn}")
launch_result

In [ ]:
# Wait for the A2A runtime to reach READY.
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]

print(f"Initial status: {status}")
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)

print(f"\nFinal status: {status}")
print(f"Agent ID : {launch_result.agent_id}")
print(f"Agent ARN: {launch_result.agent_arn}")
print("The memory-enhanced A2A agent is live. Use the bearer token from Step 6c to invoke it.")
status

## Cleanup (optional)

The memory resource created above incurs storage cost while it exists. Uncomment the cell below to delete it when you're done with this lab. Deleting the memory resource only removes memory — it does not touch the deployed runtime.

In [ ]:
# Uncomment to delete the AgentCore Memory resource created in this lab.

# memory_client.delete_memory_and_wait(memory_id=memory_id)
# print(f"Deleted memory: {memory_id}")

## Summary

In this lab you gave the warehouse agent memory and redeployed it:

- **Short-term memory** maintains session continuity (no repeated context)
- **Long-term semantic memory** remembers operational facts across sessions
- **Long-term user-preference memory** personalizes responses (formatting, thresholds)
- Memory is integrated via Strands hooks using the `MemorySessionManager` pattern
- You **deployed a new A2A-protocol AgentCore Runtime** with memory, scoping memory per conversation via `context_id` through the `agent_factory` pattern

The deployed A2A runtime is now memory-enabled. **Next: [Lab 9 — Evaluating the Warehouse Agent](09-agentcore-evaluation.ipynb)** builds an evaluation suite with Strands Evals (local) and AgentCore Evaluations (deployed).

